<a href="https://www.kaggle.com/code/mrafraim/dl-day-59-fine-tuning-a-pretrained-yolo-model?scriptVersionId=327982796" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Day 59: Fine-Tuning a Pretrained YOLO Model

Welcome to Day 59!


You'll Learn Today:

1. Why fine-tuning is preferred over training from scratch
2. Transfer learning in object detection
3. YOLO dataset structure and configuration
4. Loading pretrained YOLO weights
5. Training on a custom dataset
6. Learning rate and batch size tuning
7. Evaluating fine-tuned models
8. Common mistakes and debugging strategies

By the end of today, you should be able to:

✔ Prepare a YOLO dataset

✔ Load pretrained YOLO weights

✔ Fine-tune on a custom dataset

✔ Evaluate model performance

✔ Explain why transfer learning works


If you found this notebook helpful, your <b style="color:skyblue;">UPVOTE</b> would be greatly appreciated! It helps others discover the work and supports continuous improvement.

---

# Why Fine-Tuning Exists

Training an object detector from scratch is expensive.

The model must learn:

- Edges
- Corners
- Textures
- Shapes
- Object parts
- Full objects

This requires:

- Large datasets
- Long training times
- Significant compute resources

Instead, we start from a pretrained model.

A pretrained YOLO model already understands general visual patterns.

Fine-tuning means:

Keep most of that visual knowledge

↓

Adapt it to your specific dataset

This approach is called:

**Transfer Learning**

# Training From Scratch vs Fine-Tuning

| Aspect | Training From Scratch | Fine-Tuning |
|----------|----------|----------|
| Data Requirement | Very High | Low–Medium |
| Training Time | Long | Short |
| Compute Cost | High | Lower |
| Risk of Failure | High | Low |
| Recommended for Beginners | No | Yes |

In practice:

> Most real-world projects begin with pretrained weights.

**Example**

Suppose YOLO was originally trained on COCO.

COCO contains classes such as:

- Person
- Car
- Bicycle
- Dog
- Cat

Your task:

Detect Hard Hat of Workers.

Instead of learning vision from zero,

YOLO reuses existing visual features and learns:

"These shapes correspond to my new classes."

This dramatically reduces training time.

# Today's Goal

We will take:

Pretrained YOLOv8

and adapt it to:

> Hard Hat Detection Dataset

Classes:

1. Helmet
2. Head
3. Person

This is how object detection is commonly done in industry.

# Dataset Overview

Dataset:

> Hard Hat Detection

Kaggle Link:

https://www.kaggle.com/datasets/andrewmvd/hard-hat-detection

Dataset contains:

✔ ~5000 images

✔ Construction workers

✔ Helmets

✔ Heads

✔ Persons

Annotation Format:

>Pascal VOC XML
>
>NOT YOLO format

## Important Reality Check

The dataset is NOT ready for YOLO training.

Current format:

> images/
> annotations/

Each annotation is stored as:

> image_001.xml

YOLO expects:

> image_001.txt

Therefore:

VOC XML

↓
      
YOLO TXT


conversion is required.

---

<p style="text-align:center; color:green; font-size:18px;">(Optional)</p> 

### Understanding Pascal VOC Annotations

Example XML:

```xml

<annotation>
    <filename>image001.jpg</filename>
    <size>
        <width>640</width>
        <height>480</height>
        <depth>3</depth>
    </size>
    <object>
        <name>dog</name>
        <bndbox>
            <xmin>48</xmin>
            <ymin>240</ymin>
            <xmax>300</xmax>
            <ymax>400</ymax>
        </bndbox>
    </object>
</annotation>

```

This stores:

- Whole Image size
- Class name
- Bounding box coordinates

### Understanding YOLO Annotations

YOLO format:

`class_id x_center y_center width height`

Example:

`0 0.45 0.38 0.20 0.30`

Important:

Coordinates are normalized between 0 and 1.

Example with different object sizes and classes on a standard HD image (1920 pixels wide by 1080 pixels tall):

`0 0.15 0.30 0.10 0.20`
`1 0.75 0.65 0.30 0.40`

**Breakdown of Line 1 (Class 0: Person)**

* 0: Object Class (e.g., Person).
* 0.15: Center is 15% from the left edge.
* 0.30: Center is 30% from the top edge.
* 0.10: Box takes up 10% of the image width.
* 0.20: Box takes up 20% of the image height. [1, 2, 3, 4] 

**Breakdown of Line 2 (Class 1: Car)**

* 1: Object Class (e.g., Car).
* 0.75: Center is 75% from the left edge (on the right side).
* 0.65: Center is 65% from the top edge (near the bottom).
* 0.30: Box takes up 30% of the image width (a larger object).
* 0.40: Box takes up 40% of the image height. [5, 6] 

---

## Expected YOLO Dataset Structure

dataset/

├── images/

│   ├── train/

│   └── val/

│

├── labels/

│   ├── train/

│   └── val/

│

└── data.yaml

YOLO automatically matches:

> image.jpg
> image.txt

## Understanding YOLO Labels

Each annotation file contains:

`class_id x_center y_center width height`

Example:

0 0.51 0.42 0.15 0.20

Meaning:

Class = 0

Bounding box center = (0.51, 0.42)

Width = 15% of image

Height = 20% of image

Coordinates are normalized between 0 and 1.

## `data.yaml` Configuration File

A `data.yaml` file is just a text file used to give instructions to a computer program (usually an AI or machine learning model).

It acts like a map that tells the computer two main things:

- **Where to find pictures:** The folder paths for your training and testing images.
- **What to look for:** A list of the objects you want the computer to recognize.

If you open a data.yaml file in a text editor, it looks like this:

```yaml

# 1. Where the pictures are stored
train: ./dataset/train_images
val: ./dataset/validation_images

# 2. How many things you want to find
nc: 3                                  # nc stands for Number of Classes.

# 3. The names of those things
names: ['cat', 'dog', 'bird']

```

# Step 1 - Install YOLO

If using Kaggle or Colab:

In [1]:
!pip install ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.5 MB/s eta 0:00:00


# Step 2 - Import Libraries

The ultralytics package provides pretrained models and training utilities.

In [2]:
# Imports the YOLO class from the Ultralytics library to load, train, and run YOLO models
from ultralytics import YOLO

# Imports the built-in operating system module to interact with files, folders, and paths
import os

# Imports the shell utilities module used for high-level file operations like copying or moving files
import shutil


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


# Step 3 - Prepare Dataset

Download:

> Hard Hat Detection

from Kaggle.

Upload dataset to notebook.

After extraction:

dataset/

├── annotations/

├── images/

Verify files before proceeding.

# Step 4 - Convert VOC XML to YOLO

Because dataset annotations are XML,
we must convert them to YOLO format.

You may:

Option A:
Use a conversion script

Option B:
Use Roboflow conversion tools

Goal:

`annotations.xml`

↓

`labels.txt`

## 4.1 Understand Dataset Structure

In [3]:
DATASET_PATH = "/kaggle/input/datasets/mrafraim/hard-hat-dataset/hard_hat_dataset"

print(os.listdir(DATASET_PATH))

['annotations', 'images']


## 4.2 Define Class Mapping

In [4]:
class_map = {
    "helmet": 0,
    "head": 1,
    "person": 2
}

## 4.3 Conversion Logic

YOLO format requires normalization:

- `x_center = (xmin + xmax) / 2 / image_width`
- `y_center = (ymin + ymax) / 2 / image_height`
- `width    = (xmax - xmin) / image_width`
- `height   = (ymax - ymin) / image_height`

## 4.4 Full Conversion Script

In [5]:
# Create output folder

output_label_dir = "/kaggle/working/labels"
os.makedirs(output_label_dir, exist_ok=True)

In [6]:
# XML → YOLO Converter

import xml.etree.ElementTree as ET

def convert_voc_to_yolo(xml_file, output_file, class_map):

    tree = ET.parse(xml_file)
    root = tree.getroot()

    size = root.find("size")
    img_w = int(size.find("width").text)
    img_h = int(size.find("height").text)

    yolo_lines = []

    for obj in root.findall("object"):
        class_name = obj.find("name").text

        if class_name not in class_map:
            continue

        class_id = class_map[class_name]

        bndbox = obj.find("bndbox")
        xmin = float(bndbox.find("xmin").text)
        ymin = float(bndbox.find("ymin").text)
        xmax = float(bndbox.find("xmax").text)
        ymax = float(bndbox.find("ymax").text)

        # YOLO conversion
        x_center = (xmin + xmax) / 2.0 / img_w
        y_center = (ymin + ymax) / 2.0 / img_h
        width = (xmax - xmin) / img_w
        height = (ymax - ymin) / img_h

        yolo_lines.append(
            f"{class_id} {x_center} {y_center} {width} {height}"
        )

    with open(output_file, "w") as f:
        f.write("\n".join(yolo_lines))

In [7]:
# Batch Convert All XML Files

xml_dir = "/kaggle/input/datasets/mrafraim/hard-hat-dataset/hard_hat_dataset/annotations"

for xml_file in os.listdir(xml_dir):

    if not xml_file.endswith(".xml"):
        continue

    full_xml_path = os.path.join(xml_dir, xml_file)

    output_txt_path = os.path.join(
        output_label_dir,
        xml_file.replace(".xml", ".txt")
    )

    convert_voc_to_yolo(full_xml_path, output_txt_path, class_map)

In [8]:
print(os.listdir(output_label_dir)[:5])

['hard_hat_workers4958.txt', 'hard_hat_workers1308.txt', 'hard_hat_workers646.txt', 'hard_hat_workers3660.txt', 'hard_hat_workers1447.txt']


# Step 5 - Train / Validation Split

## Step 5.1 - Define Paths

In [9]:
import random

images_path = "/kaggle/input/datasets/mrafraim/hard-hat-dataset/hard_hat_dataset/images"
labels_path = "/kaggle/working/labels"

output_base = "/kaggle/working/dataset"

train_img_dir = os.path.join(output_base, "images/train")
val_img_dir   = os.path.join(output_base, "images/val")

train_lbl_dir = os.path.join(output_base, "labels/train")
val_lbl_dir   = os.path.join(output_base, "labels/val")

# create folders
for path in [train_img_dir, val_img_dir, train_lbl_dir, val_lbl_dir]:
    os.makedirs(path, exist_ok=True)

## Step 5.2 - Get All Images

In [10]:
image_files = [f for f in os.listdir(images_path) if f.endswith(".png")]

print("Total images:", len(image_files))

Total images: 5000


## Step 5.3 - Shuffle Dataset

Prevents biased splits (e.g., similar images grouped together)

In [11]:
random.seed(42)
random.shuffle(image_files)

## Step 5.4 - Train / Val Split (80/20)

In [12]:
split_ratio = 0.8

train_size = int(len(image_files) * split_ratio)

train_files = image_files[:train_size]
val_files   = image_files[train_size:]

print("Train:", len(train_files))
print("Val:", len(val_files))

Train: 4000
Val: 1000


## Step 5.5 - Copy Files Properly

We move BOTH:

- image
- corresponding label

In [13]:
# train set

for img in train_files:

    img_path = os.path.join(images_path, img)
    lbl_path = os.path.join(labels_path, img.replace(".png", ".txt"))

    shutil.copy(img_path, train_img_dir)
    
    if os.path.exists(lbl_path):
        shutil.copy(lbl_path, train_lbl_dir)

In [14]:
# validation set 

for img in val_files:

    img_path = os.path.join(images_path, img)
    lbl_path = os.path.join(labels_path, img.replace(".png", ".txt"))

    shutil.copy(img_path, val_img_dir)
    
    if os.path.exists(lbl_path):
        shutil.copy(lbl_path, val_lbl_dir)

## Step 5.6 - Sanity Check

In [15]:
print("Train images:", len(os.listdir(train_img_dir)))
print("Val images:", len(os.listdir(val_img_dir)))

Train images: 4000
Val images: 1000


In [16]:
print("Train labels:", len(os.listdir(train_lbl_dir)))
print("Val labels:", len(os.listdir(val_lbl_dir)))

Train labels: 4000
Val labels: 1000


<p style="text-align:center; color:red; font-size:18px;"> To be continue...</p>

---

<p style="text-align:center; color:skyblue; font-size:18px;">
© 2026 Mostafizur Rahman
</p>
